# Keras fraud classifier

Run in a fresh environment with the TensorFlow requirements. The public repository needs no GitHub token.


## Execution provenance

All code cells below were executed on 16 November 2026 in an isolated CPU checkout. These are fresh verification-training outputs, recorded in `results/verification/2026-11-16/keras.json`. The published report and slides use the earlier frozen `results/keras.json`; the rerun did not overwrite that experiment. Different software, hardware and training randomness can change the metrics. See `docs/demo_readiness.md` for the comparison.


## Setup and shared data

In [1]:
from pathlib import Path
import os, subprocess
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (repo_root / "src" / "data.py").exists():
    repo_root = Path("/content/credit-card-fraud-detection")
    if not (repo_root / "src" / "data.py").exists():
        subprocess.run(["git", "clone", "https://github.com/VanshSharmaPES/credit-card-fraud-detection.git", str(repo_root)], check=True)
os.chdir(repo_root)
print("Project root:", Path.cwd())


Project root: /home/vanshsharma/Documents/Codex/2026-11-15/handoff-credit-card-fraud-detection-user/work/finalization/runrepo


In [2]:
%pip install -q -r requirements/base.txt
%pip install -q -r requirements/keras.txt
import subprocess, sys
subprocess.run([sys.executable, "src/data.py"], check=True)
# In the full team repository, run the existing setup/data cells first.
# The team's src/data.py must expose load_splits() as agreed in the project plan.
from pathlib import Path
import sys, random, time, platform
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path.cwd() / "src"))
from data import load_splits
from evaluate import evaluate_model, permutation_importance
SEED = 42
random.seed(SEED); np.random.seed(SEED)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(SEED)
(X_train, y_train, X_val, y_val, X_test, y_test,
 row_id_val, row_id_test, amount_val, amount_test) = load_splits()
feature_names = list(X_train.columns)
X_train = np.asarray(X_train, dtype=np.float32)
X_val = np.asarray(X_val, dtype=np.float32)
X_test = np.asarray(X_test, dtype=np.float32)
y_train = np.asarray(y_train, dtype=np.int64)
y_val = np.asarray(y_val, dtype=np.int64)
y_test = np.asarray(y_test, dtype=np.int64)
class_weight = {0: 1.0, 1: float((y_train == 0).sum() / (y_train == 1).sum())}



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: /home/vanshsharma/Documents/Codex/2026-11-15/handoff-credit-card-fraud-detection-user/work/finalization/tf-venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: /home/vanshsharma/Documents/Codex/2026-11-15/handoff-credit-card-fraud-detection-user/work/finalization/tf-venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Traceback (most recent call last):
  File "/home/vanshsharma/Documents/Codex/2026-11-15/handoff-credit-card-fraud-detection-user/work/finalization/runrepo/src/data.py", line 10, in <module>
    import pandas as pd
ModuleNotFoundError: No module named 'pandas'


I0000 00:00:1794834348.828143   20517 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794834368.451164   20517 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1794834384.339176   20517 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## Train and evaluate

In [3]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(64, activation="relu"), layers.Dropout(0.2),
    layers.Dense(32, activation="relu"), layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[keras.metrics.AUC(curve="PR", name="pr_auc")],
)
callbacks = [keras.callbacks.EarlyStopping(
    monitor="val_pr_auc", mode="max", patience=5, restore_best_weights=True
)]
start = time.perf_counter()
history = model.fit(
    X_train, y_train, validation_data=(X_val, y_val), class_weight=class_weight,
    batch_size=2048, epochs=30, callbacks=callbacks, verbose=1,
)
train_seconds = time.perf_counter() - start
p_val = model.predict(X_val, batch_size=8192, verbose=0).ravel()
start = time.perf_counter()
p_test = model.predict(X_test, batch_size=8192, verbose=0).ravel()
predict_seconds = time.perf_counter() - start
Path("models").mkdir(exist_ok=True)
model.save("models/keras.keras")
result = evaluate_model(
    "keras", y_val, p_val, y_test, p_test, row_id_val, row_id_test,
    train_seconds, predict_seconds,
    notes="Dense(64,32) ReLU network; dropout 0.2; class weighted; early stopped on validation PR-AUC.",
    hardware=f"{platform.machine()} CPU",
)
result


E0000 00:00:1794834389.642643   20517 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
I0000 00:00:1794834389.642758   20517 cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES="-1"
I0000 00:00:1794834389.647509   20517 cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
I0000 00:00:1794834389.647550   20517 cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
I0000 00:00:1794834389.647554   20517 cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: fedora
I0000 00:00:1794834389.647561   20517 cuda_diagnostics.cc:183] hostname: fedora
I0000 00:00:1794834389.647839   20517 cuda_diagnostics.cc:190] libcuda reported version is: 615.78.8
I0000 00:00:1794834389.647899   20517 cuda_diagnostics.cc:194] kernel reported 

Epoch 1/30


 1/84 ━━━━━━━━━━━━━━━━━━━━ 2:49 2s/step - loss: 3.7801 - pr_auc: 0.0415

 7/84 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.4802 - pr_auc: 0.0701 

18/84 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.1529 - pr_auc: 0.1970

29/84 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.0420 - pr_auc: 0.2836

... (63 outputs left out)

74/84 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2181 - pr_auc: 0.6878

82/84 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2124 - pr_auc: 0.6789

84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.2109 - pr_auc: 0.6760 - val_loss: 0.0819 - val_pr_auc: 0.6632


{'framework': 'keras',
 'threshold': 0.9603972434997559,
 'precision': 0.8043478260869565,
 'recall': 0.7789473684210526,
 'f1': 0.7914438502673797,
 'pr_auc': 0.7380379708361576,
 'roc_auc': 0.9621689216244615,
 'tn': 56633,
 'fp': 18,
 'fn': 21,
 'tp': 74,
 'rmse': 0.17924741860922558,
 'mae': 0.12821787892635034,
 'r2': -18.22406125986268,
 'train_seconds': 6.606101316001514,
 'predict_seconds': 0.13627580600041256,
 'hardware': 'x86_64 CPU',
 'notes': 'Dense(64,32) ReLU network; dropout 0.2; class weighted; early stopped on validation PR-AUC.'}

## Validation permutation importance

Run this as an optional transparency output. It uses validation data and average precision; it does not change the trained model.

In [4]:
def keras_predict(frame):
    arr = np.asarray(frame, dtype=np.float32)
    return model.predict(arr, batch_size=8192, verbose=0).ravel()
perm = permutation_importance(keras_predict, pd.DataFrame(X_val, columns=feature_names), y_val, n_repeats=3, random_state=SEED)
Path("results").mkdir(exist_ok=True)
perm.to_csv("results/perm_keras.csv", index=False)
perm.head(10)

,feature,importance_mean,importance_std,baseline_pr_auc
0,V14,0.026686,0.002807,0.689137
1,V5,0.013928,0.000812,0.689137
2,V8,0.010580,0.006717,0.689137
3,V17,0.005279,0.001259,0.689137
4,V16,0.004661,0.003059,0.689137
5,V27,0.004411,0.003341,0.689137
6,V28,0.003623,0.003654,0.689137
7,V19,0.002713,0.001806,0.689137
8,V15,0.002272,0.005870,0.689137
9,Hour,0.001611,0.001141,0.689137
